### Note about coordinates for mapping
- In this case study, **Soil Chem** and **Soil Temp** tabs may **not** include latitude/longitude columns.
- This notebook will **backfill** coordinates by matching `site_id` to another tab that *does* have coordinates (often **Water Chem**), and it will also fill the four GLORIA summits (**US_YNP_BLP/SCP/SPP/WSP**) from GLORIA metadata.
- Always sanity-check your map: use the `coord_flag` and `coord_source_tab` columns.


# GEOL230 — YNP Case Study: ArcGIS Export (Student Starter)
This version includes **checkpoints + TODOs** so you understand what’s being exported for ArcGIS.
Use this notebook to create **GIS-ready exports** from your cleaned Excel workbook.

**Outputs created in `outputs/`:**
- `Map1_points_for_arcgis.csv` (required by the assignment)
- `Map1_points_all_records.csv` (optional, includes all rows with coordinates)
- `Map1_points_unique_sites.csv` (optional, one point per site per dataset)
- `Map1_points_readme.txt` (field descriptions + quick ArcGIS tips)

> If your cleaned tabs do not include coordinates, you must obtain coordinates from dataset metadata (and document how) or choose a dataset/tab that includes coordinates.


## 1) Setup

In [ ]:
# STUDENT TASK: Get your cleaned Excel workbook into this Colab session.
# Option A (recommended): Upload your Excel file manually
from google.colab import files
uploaded = files.upload()  # choose your LASTNAME_FIRSTNAME_GEOL230_YNP_CaseStudy.xlsx

# Option B (optional): Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# Then set EXCEL_PATH below to the full path in your Drive (e.g., /content/drive/MyDrive/...)


In [ ]:
                    import os
                    import re
                    import numpy as np
                    import pandas as pd

                    # STUDENT TASK (only if needed):
                    # If the notebook can't find your workbook automatically, set EXCEL_PATH to your filename.
                    # Example:
                    # EXCEL_PATH = "Smith_Alex_GEOL230_YNP_CaseStudy.xlsx"
                    EXCEL_PATH = None

                    def guess_excel_path():
                        # pick the newest .xlsx in the current folder if EXCEL_PATH not set
                        xlsx = [f for f in os.listdir('.') if f.lower().endswith('.xlsx')]
                        if not xlsx:
                            return None
                        xlsx.sort(key=lambda f: os.path.getmtime(f), reverse=True)
                        return xlsx[0]

                    if EXCEL_PATH is None:
                        EXCEL_PATH = guess_excel_path()

                    if EXCEL_PATH is None or not os.path.exists(EXCEL_PATH):
                        raise FileNotFoundError(
                            "Couldn't find your Excel workbook.
"
                            "- In Colab: run the upload cell again and select your .xlsx file.
"
                            "- Or set EXCEL_PATH to the correct filename/path."
                        )

                    print("Using Excel workbook:", EXCEL_PATH)

                    os.makedirs("outputs", exist_ok=True)


## 2) Load your cleaned tabs

In [ ]:
# STUDENT CHECKPOINT:
# These are the expected clean sheet names from the assignment.
# If your workbook sheet names are different, update the strings below to match EXACTLY.
SHEETS = {
    "soil_chem": "Clean Soil Chem (YNP)",
    "soil_temp": "Clean Soil Temp (YNP)",
    "water_chem": "Clean Water Chem (YNP)",
}

xl = pd.ExcelFile(EXCEL_PATH)

print("Sheets found in workbook:")
for s in xl.sheet_names:
    print(" -", s)

def read_sheet(sheet_name):
    if sheet_name not in xl.sheet_names:
        raise ValueError(
            f"Sheet not found: '{sheet_name}'.\n"
            "Fix: check spelling/capitalization in your workbook OR update the SHEETS dict above."
        )
    return pd.read_excel(EXCEL_PATH, sheet_name=sheet_name)

soil_chem_df = read_sheet(SHEETS["soil_chem"])
soil_temp_df = read_sheet(SHEETS["soil_temp"])
water_chem_df = read_sheet(SHEETS["water_chem"])

print("\nLoaded shapes:")
print("soil_chem_df:", soil_chem_df.shape)
print("soil_temp_df:", soil_temp_df.shape)
print("water_chem_df:", water_chem_df.shape)


## 3) Find latitude/longitude columns (auto-detect)

In [ ]:
# Many public datasets use different coordinate field names.
# This helper searches for likely latitude/longitude columns (case-insensitive).

LAT_CANDIDATES = [
    "latitude", "lat", "y", "dec_lat_va", "dec_lat", "lat_dd", "y_coord", "ycoord",
    "site_lat", "station_lat", "latitudedegrees", "lat_deg", "lat_dd_wgs84"
]
LON_CANDIDATES = [
    "longitude", "lon", "long", "lng", "x", "dec_long_va", "dec_long", "lon_dd", "x_coord", "xcoord",
    "site_lon", "site_long", "station_lon", "longitudedegrees", "lon_deg", "lon_dd_wgs84"
]

def normalize_colname(c):
    return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())

def find_coord_cols(df):
    cols = list(df.columns)
    norm = {c: normalize_colname(c) for c in cols}
    inv = {v: c for c, v in norm.items()}  # last wins, that's OK for our uses

    lat_col = None
    lon_col = None

    # exact matches to candidate normalized names
    lat_norms = [normalize_colname(x) for x in LAT_CANDIDATES]
    lon_norms = [normalize_colname(x) for x in LON_CANDIDATES]

    for n in lat_norms:
        if n in inv:
            lat_col = inv[n]
            break
    for n in lon_norms:
        if n in inv:
            lon_col = inv[n]
            break

    # fallback: look for 'lat'/'lon' substrings
    if lat_col is None:
        for c in cols:
            n = norm[c]
            if "lat" in n and "plate" not in n:
                lat_col = c
                break
    if lon_col is None:
        for c in cols:
            n = norm[c]
            if ("lon" in n) or ("long" in n and "longitud" in n):
                lon_col = c
                break

    return lat_col, lon_col

for name, df in [("soil_chem_df", soil_chem_df), ("soil_temp_df", soil_temp_df), ("water_chem_df", water_chem_df)]:
    lat, lon = find_coord_cols(df)
    print(f"{name}: latitude={lat!r}, longitude={lon!r}")


### Optional: override column choices (only if auto-detect is wrong)
If the notebook guessed the wrong columns, fill in the dictionaries below.
Leave values as `None` to keep auto-detection.


In [ ]:
# Fill these ONLY if needed.
# Example:
# OVERRIDE_COORDS[SHEETS["soil_chem"]] = {"lat": "dec_lat_va", "lon": "dec_long_va"}
# OVERRIDE_SITE[SHEETS["soil_chem"]]   = "site_id"
# OVERRIDE_DATE[SHEETS["soil_chem"]]   = "sample_date"

OVERRIDE_COORDS = {
    SHEETS["soil_chem"]: {"lat": None, "lon": None},
    SHEETS["soil_temp"]: {"lat": None, "lon": None},
    SHEETS["water_chem"]: {"lat": None, "lon": None},
}

OVERRIDE_SITE = {
    SHEETS["soil_chem"]: None,
    SHEETS["soil_temp"]: None,
    SHEETS["water_chem"]: None,
}

OVERRIDE_DATE = {
    SHEETS["soil_chem"]: None,
    SHEETS["soil_temp"]: None,
    SHEETS["water_chem"]: None,
}


## 4) Build GIS-ready point tables

In [ ]:
def to_float(series):
    # Convert to float safely (removes commas, strips whitespace)
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

SITE_CANDIDATES = [
    "site", "site_id", "siteid", "plot", "plot_id", "plotid", "station", "station_id", "stationid",
    "summit", "location", "loc", "name", "sitename", "site_name"
]
DATE_CANDIDATES = ["date", "sample_date", "datetime", "time", "collection_date", "collectdate"]

def find_best_col(df, candidates):
    cols = list(df.columns)
    norm = {c: normalize_colname(c) for c in cols}
    cand_norm = [normalize_colname(x) for x in candidates]
    for cn in cand_norm:
        for c in cols:
            if norm[c] == cn:
                return c
    # fallback: substring
    for c in cols:
        n = norm[c]
        for cn in cand_norm:
            if cn in n:
                return c
    return None

def build_points(df, source_name, overrides=None):
    overrides = overrides or {}
    lat_col, lon_col = find_coord_cols(df)

    # Apply overrides if you filled them in above
    lat_col = overrides.get("lat") or lat_col
    lon_col = overrides.get("lon") or lon_col

    if lat_col is None or lon_col is None:
        # Allow missing coordinate columns (common in this case study for Soil Chem / Soil Temp).
        # We will create empty Latitude/Longitude and fill later from other tabs and/or GLORIA metadata.
        lat_col = None
        lon_col = None

    site_col = overrides.get("site") or find_best_col(df, SITE_CANDIDATES)
    date_col = overrides.get("date") or find_best_col(df, DATE_CANDIDATES)

    print(f"[{source_name}] Using columns → site: {site_col!r}, date: {date_col!r}, lat: {lat_col!r}, lon: {lon_col!r}")

    out = pd.DataFrame()
    out["source_tab"] = source_name
    out["coord_source_tab"] = source_name
    out["site_id"] = df[site_col].astype(str).str.strip() if site_col else "UNKNOWN_SITE"

    if date_col:
        out["date"] = pd.to_datetime(df[date_col], errors="coerce")
    else:
        out["date"] = pd.NaT

    if lat_col is not None and lon_col is not None:
        out["Latitude"] = to_float(df[lat_col])
        out["Longitude"] = to_float(df[lon_col])
    else:
        out["Latitude"] = np.nan
        out["Longitude"] = np.nan

    # keep a few extra descriptive columns if they exist
    for extra in ["Type", "type", "area", "Area", "feature", "Feature", "Parameter", "parameter"]:
        if extra in df.columns:
            out[extra] = df[extra]

    # basic coordinate sanity check (Yellowstone area roughly lat 44–45.2, lon -111.5 to -109.8)
    out["coord_flag"] = ""
    bad = out["Latitude"].isna() | out["Longitude"].isna()
    if lat_col is None or lon_col is None:
        out.loc[bad, "coord_flag"] = "missing_no_columns"
    else:
        out.loc[bad, "coord_flag"] = "missing"

    out_of_range = (~bad) & ((out["Latitude"] < -90) | (out["Latitude"] > 90) | (out["Longitude"] < -180) | (out["Longitude"] > 180))
    out.loc[out_of_range, "coord_flag"] = "out_of_range"

    y_box = (~bad) & ((out["Latitude"] < 43.5) | (out["Latitude"] > 46.0) | (out["Longitude"] < -112.5) | (out["Longitude"] > -108.8))
    out.loc[y_box & (out["coord_flag"] == ""), "coord_flag"] = "check_location"

    return out

# STUDENT CHECKPOINT: If any of the printed column choices look wrong,
# fill in OVERRIDE_COORDS / OVERRIDE_SITE / OVERRIDE_DATE above and re-run this cell.

soil_points = build_points(
    soil_chem_df,
    SHEETS["soil_chem"],
    overrides={
        "lat": OVERRIDE_COORDS[SHEETS["soil_chem"]]["lat"],
        "lon": OVERRIDE_COORDS[SHEETS["soil_chem"]]["lon"],
        "site": OVERRIDE_SITE[SHEETS["soil_chem"]],
        "date": OVERRIDE_DATE[SHEETS["soil_chem"]],
    } if "OVERRIDE_COORDS" in globals() else None
)

temp_points = build_points(
    soil_temp_df,
    SHEETS["soil_temp"],
    overrides={
        "lat": OVERRIDE_COORDS[SHEETS["soil_temp"]]["lat"],
        "lon": OVERRIDE_COORDS[SHEETS["soil_temp"]]["lon"],
        "site": OVERRIDE_SITE[SHEETS["soil_temp"]],
        "date": OVERRIDE_DATE[SHEETS["soil_temp"]],
    } if "OVERRIDE_COORDS" in globals() else None
)

water_points = build_points(
    water_chem_df,
    SHEETS["water_chem"],
    overrides={
        "lat": OVERRIDE_COORDS[SHEETS["water_chem"]]["lat"],
        "lon": OVERRIDE_COORDS[SHEETS["water_chem"]]["lon"],
        "site": OVERRIDE_SITE[SHEETS["water_chem"]],
        "date": OVERRIDE_DATE[SHEETS["water_chem"]],
    } if "OVERRIDE_COORDS" in globals() else None
)

points_all = pd.concat([soil_points, temp_points, water_points], ignore_index=True)


# --- GLORIA summit coordinate backfill (Yellowstone: US_YNP_BLP/SCP/SPP/WSP) ---
# Some tabs in the case study may not include coordinates for the GLORIA summit sites.
# We can safely fill them from the NPS GLORIA site metadata (decimal degrees).
GLORIA_SUMMIT_COORDS = {
    "US_YNP_BLP": (44.70028, -109.8267),
    "US_YNP_SCP": (44.69584, -109.8339),
    "US_YNP_SPP": (44.70167, -109.8350),
    "US_YNP_WSP": (44.69444, -109.8364),
}

def backfill_gloria_coords(points_df, site_col="site_id", lat_col="Latitude", lon_col="Longitude"):
    site_key = points_df[site_col].astype(str).str.strip().str.upper()
    lat_map = {k: v[0] for k, v in GLORIA_SUMMIT_COORDS.items()}
    lon_map = {k: v[1] for k, v in GLORIA_SUMMIT_COORDS.items()}

    needs_lat = site_key.isin(lat_map) & points_df[lat_col].isna()
    needs_lon = site_key.isin(lon_map) & points_df[lon_col].isna()

    points_df.loc[needs_lat, lat_col] = site_key.map(lat_map)[needs_lat]
    points_df.loc[needs_lon, lon_col] = site_key.map(lon_map)[needs_lon]

    filled_mask = (needs_lat | needs_lon)

    # If a coord was missing and is now filled, update the flag so it's transparent
    if "coord_flag" in points_df.columns:
        fixed = site_key.isin(lat_map) & points_df[lat_col].notna() & points_df[lon_col].notna()
        points_df.loc[fixed & (points_df["coord_flag"].isin(["missing","missing_no_columns"])), "coord_flag"] = "gloria_filled"

    # Track coordinate provenance
    if "coord_source_tab" in points_df.columns:
        points_df.loc[filled_mask, "coord_source_tab"] = "GLORIA_metadata"

    return points_df

points_all = backfill_gloria_coords(points_all)
# ------------------------------------------------------------------------------

# --- Fill missing coordinates using any other tab that DOES have coordinates ---
# For this case study, Soil Chem and Soil Temp often lack Lat/Lon fields.
# We can safely backfill those sites using coordinates from tabs that include them (e.g., Water Chem).
def fill_missing_coords_from_lookup(points_df, site_col="site_id", lat_col="Latitude", lon_col="Longitude"):
    pts = points_df.copy()
    site_key = pts[site_col].astype(str).str.strip().str.upper()
    pts["_site_key"] = site_key

    # Build a lookup from rows that already have valid coords
    have = pts.dropna(subset=[lat_col, lon_col]).copy()
    if have.empty:
        pts.drop(columns=["_site_key"], inplace=True)
        return pts

    lookup = (
        have
        .sort_values(["source_tab", site_col])
        .groupby("_site_key", as_index=False)
        .agg({lat_col: "first", lon_col: "first", "source_tab": "first"})
    )

    lat_map = dict(zip(lookup["_site_key"], lookup[lat_col]))
    lon_map = dict(zip(lookup["_site_key"], lookup[lon_col]))
    src_map = dict(zip(lookup["_site_key"], lookup["source_tab"]))

    needs = pts[lat_col].isna() | pts[lon_col].isna()
    can_fill = needs & site_key.isin(lat_map)

    pts.loc[can_fill, lat_col] = site_key.map(lat_map)[can_fill]
    pts.loc[can_fill, lon_col] = site_key.map(lon_map)[can_fill]

    # Update flags + provenance transparently
    if "coord_flag" in pts.columns:
        pts.loc[can_fill & (pts["coord_flag"].isin(["missing","missing_no_columns",""])), "coord_flag"] = "filled_from_other_tab"

    if "coord_source_tab" in pts.columns:
        pts.loc[can_fill, "coord_source_tab"] = site_key.map(src_map)[can_fill]

    pts.drop(columns=["_site_key"], inplace=True)
    return pts

points_all = fill_missing_coords_from_lookup(points_all)
# ------------------------------------------------------------------------------
points_all["Latitude"] = points_all["Latitude"].round(6)
points_all["Longitude"] = points_all["Longitude"].round(6)

print("points_all:", points_all.shape)
print("Unique site_ids (all tabs):", points_all["site_id"].nunique())
points_all.head()


## 5) Create a **unique-sites** layer for mapping (recommended)

In [ ]:
# Many tables have multiple rows per site (especially daily temperature).
# For mapping, you usually want ONE point per site.

points_unique = (
    points_all
    .dropna(subset=["Latitude", "Longitude"])
    .sort_values(["source_tab", "site_id"])
    .groupby(["source_tab", "site_id"], as_index=False)
    .agg({
        "Latitude": "first",
        "Longitude": "first",
        "date": "min",
        "coord_flag": "first"
    })
)

print("points_unique:", points_unique.shape)
points_unique.head()


### Checkpoint: do these look reasonable?
- How many points per tab?
- Any `coord_flag` values other than blank?


In [ ]:
# Count points by tab and coordinate flag
display(points_unique.groupby(["source_tab", "coord_flag"]).size().reset_index(name="n"))

# Quick peek at any flagged points (if none, this will be empty)
flagged = points_unique[points_unique["coord_flag"] != ""]
print("Flagged points (if any):", flagged.shape)
flagged.head(10)


## 6) Export to CSV for ArcGIS

In [ ]:
# Required filename for the assignment:
required_path = "outputs/Map1_points_for_arcgis.csv"

# Also export optional files
all_records_path = "outputs/Map1_points_all_records.csv"
unique_sites_path = "outputs/Map1_points_unique_sites.csv"

# Export
points_unique.to_csv(required_path, index=False)
points_all.to_csv(all_records_path, index=False)
points_unique.to_csv(unique_sites_path, index=False)

print("Wrote:")
print(" -", required_path)
print(" -", all_records_path)
print(" -", unique_sites_path)

# Create a small README for ArcGIS usage
readme = """Map1_points_for_arcgis.csv — Field guide

Required fields (for Add XY Point Data in ArcGIS Pro):
- Longitude (X)
- Latitude  (Y)

Other fields:
- source_tab: which cleaned tab the record came from
- site_id: site/plot/station identifier
- date: parsed date (if present)
- coord_source_tab: where the coordinates came from (same tab, another tab, or GLORIA_metadata)
- coord_flag: QA flag for coordinates

coord_flag values:
- "" (blank) = coordinate looks fine
- missing = lat/lon columns existed but the value was missing on that row
- missing_no_columns = that tab did not include coordinate columns (will be filled later if possible)
- gloria_filled = filled from NPS GLORIA summit metadata (US_YNP_BLP/SCP/SPP/WSP)
- filled_from_other_tab = filled by matching site_id to another tab with coordinates (often Water Chem)
- out_of_range = coordinate outside valid lat/lon ranges
- check_location = outside a Yellowstone-ish bounding box (not necessarily wrong; check)

ArcGIS Pro tip:
- Use Add XY Point Data with WGS 84 (EPSG:4326) when lat/lon are decimal degrees.
"""

with open("outputs/Map1_points_readme.txt", "w", encoding="utf-8") as f:
    f.write(readme)

print(" - outputs/Map1_points_readme.txt")


## 7) (Optional) Export as GeoJSON / Shapefile (more GIS-native)

In [ ]:
# ArcGIS Pro can read CSVs directly (recommended for this assignment),
# but GeoJSON/Shapefile can be useful if you want a more 'GIS-native' output.
# This cell tries to export both. If it fails, you can skip it.

try:
    import geopandas as gpd
    from shapely.geometry import Point

    gdf = gpd.GeoDataFrame(
        points_unique.copy(),
        geometry=[Point(xy) for xy in zip(points_unique["Longitude"], points_unique["Latitude"])],
        crs="EPSG:4326"
    )

    geojson_path = "outputs/Map1_points_unique_sites.geojson"
    gdf.to_file(geojson_path, driver="GeoJSON")
    print("Wrote:", geojson_path)

    # Shapefile writes multiple files, so we put them in a folder and zip it
    shp_dir = "outputs/Map1_points_unique_sites_shp"
    os.makedirs(shp_dir, exist_ok=True)
    shp_path = os.path.join(shp_dir, "Map1_points_unique_sites.shp")
    gdf.to_file(shp_path, driver="ESRI Shapefile")
    print("Wrote shapefile folder:", shp_dir)

    # Zip the shapefile folder for easy downloading
    import zipfile
    shp_zip = "outputs/Map1_points_unique_sites_shp.zip"
    with zipfile.ZipFile(shp_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(shp_dir):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, start=os.path.dirname(shp_dir))
                z.write(full, arcname=rel)
    print("Zipped shapefile:", shp_zip)

except Exception as e:
    print("GeoJSON/Shapefile export skipped (geopandas not available or export failed).")
    print("Reason:", repr(e))
    print("\nIf you want to try anyway in Colab, uncomment and run:")
    print("!pip -q install geopandas pyogrio shapely")


## 8) (Optional) Download outputs/ as a ZIP (Colab)

In [ ]:
# If you're in Google Colab, this will create a ZIP and download it.
# (If you're not in Colab, you can ignore this cell.)

import zipfile

zip_path = "outputs/Map1_arcgis_exports.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for fn in ["Map1_points_for_arcgis.csv", "Map1_points_all_records.csv", "Map1_points_unique_sites.csv", "Map1_points_readme.txt"]:
        z.write(os.path.join("outputs", fn), arcname=os.path.join("outputs", fn))

print("Created:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Not running in Colab (download skipped).", e)
